<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-Manh2307/PDF-Rag-Assistant/blob/feature-rag/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cài ollama

In [1]:
import os, subprocess, time
!sudo apt-get update && sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh
os.environ['OLLAMA_FLASH_ATTENTION'] = 'false'
subprocess.Popen(["ollama", "serve"]); time.sleep(10)

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.7.0-6).
zstd is already the newest version (1.4.8+

In [2]:
!ollama pull vicuna:7b-v1.5-q5_1
!ollama pull bge-m3

# Install & import

In [3]:
!pip install -q pypdf chromadb ollama

In [4]:
from pypdf import PdfReader
import ollama
import chromadb

# Download file

In [14]:
# https://drive.google.com/file/d/1lWuq0COKnU9mCfMvTEq54DBLgAh3yYDx/view?usp=drive_link
!gdown 1lWuq0COKnU9mCfMvTEq54DBLgAh3yYDx

Downloading...
From: https://drive.google.com/uc?id=1lWuq0COKnU9mCfMvTEq54DBLgAh3yYDx
To: /content/PDF-Rag-Assistant/YOLOv10_Tutorials.pdf
100% 16.6M/16.6M [00:00<00:00, 17.7MB/s]


# 1. Read PDF

In [37]:
pdfPath = "YOLOv10_Tutorials.pdf"
reader = PdfReader(pdfPath)

# số trang
num_pages = len(reader.pages)

# full text extract
full_text = '\n'.join(page.extract_text() or "" for page in reader.pages)

print("Số trang: ", num_pages)
print("Tổng ký tự: ", len(full_text))



Số trang:  20
Tổng ký tự:  23157


# 2. Chunking

In [46]:
def fixed_size_chunk(text, size = 1000, overlap=200):
  # Tách text
  paras = [p.strip() for p in text.split("\n") if p.strip()]

  chunks, current_text = [], ""

  # chunking
  for p in paras:
    if len(current_text) + len(p) + 1 <= size:
      current_text += p + '\n'
    else:
      if current_text:
        chunks.append(current_text.strip())
      current_text = (current_text[-overlap:] + p + '\n') if overlap else (p + '\n')

  if current_text.strip():
    chunks.append(current_text.strip())
  return chunks

chunks = fixed_size_chunk(full_text)

print("Số chunk: ", len(chunks))
char_num = 200
print(f"Đoạn chunk text của {char_num} kí tự:\n ", chunks[0][:char_num])

Số chunk:  30
Đoạn chunk text của 200 kí tự:
  AI VIET NAM – AI COURSE 2024
Tutorial: Phát hiện đối tượng trong ảnh với
YOLOv10
Dinh-Thang Duong, Nguyen-Thuan Duong, Minh-Duc Bui và
Quang-Vinh Dinh
Ngày 20 tháng 6 năm 2024
I. Giới thiệu
Object Det
